# Lezione 1 — Forward Propagation e Funzioni di Attivazione

## Obiettivi di apprendimento
- Capire come una rete neurale calcola una previsione combinando pesi, bias e una funzione di attivazione (forward propagation).
- Implementare da zero, con NumPy puro (senza Keras né TensorFlow), una rete neurale generica strato per strato.
- Eseguire quella rete con due architetture diverse e osservarne gli output.
- Confrontare le funzioni di attivazione sigmoide e ReLU, e visualizzare in un grafico il problema del *vanishing gradient*.

## Cosa costruiremo
Programmeremo i pezzi di una rete neurale — inizializzazione dei pesi, somma pesata, attivazione — e il ciclo che li concatena strato per strato. Poi confronteremo due funzioni di attivazione e vedremo, con un grafico, perché una di esse facilita l'addestramento di reti più profonde rispetto all'altra.

## Ripasso teorico veloce: forward propagation

Una rete neurale è una catena di nodi organizzati in strati. Ogni nodo:

1. Riceve uno o più input.
2. Calcola una **somma pesata**: `z = x1*w1 + x2*w2 + ... + bias`.
3. Passa `z` attraverso una **funzione di attivazione** (qui, la sigmoide `a = 1/(1+e^-z)`).
4. Restituisce quell'output `a` come input allo strato successivo.

Ripetere questo procedimento dallo strato di input fino allo strato di output è **forward propagation**. Importante: in questa lezione i pesi sono casuali — non c'è ancora addestramento, solo il calcolo di una previsione con i pesi che la rete già possiede.

In [ ]:
import numpy as np

def initialize_network(num_inputs, num_hidden_layers, num_nodes_hidden, num_nodes_output):
    """Crea la struttura della rete: un dizionario con pesi e bias casuali per ogni nodo."""
    num_nodes_previous = num_inputs
    network = {}

    for layer in range(num_hidden_layers + 1):
        if layer == num_hidden_layers:
            layer_name = 'output'
            num_nodes = num_nodes_output
        else:
            layer_name = 'layer_{}'.format(layer + 1)
            num_nodes = num_nodes_hidden[layer]

        network[layer_name] = {}
        for node in range(num_nodes):
            node_name = 'node_{}'.format(node + 1)
            network[layer_name][node_name] = {
                'weights': np.around(np.random.uniform(size=num_nodes_previous), decimals=2),
                'bias': np.around(np.random.uniform(size=1), decimals=2),
            }

        num_nodes_previous = num_nodes

    return network


def compute_weighted_sum(inputs, weights, bias):
    """z = prodotto scalare tra input e pesi, più il bias."""
    return np.sum(inputs * weights) + bias


def node_activation(weighted_sum):
    """Attivazione sigmoide: comprime qualsiasi numero nell'intervallo (0, 1)."""
    return 1.0 / (1.0 + np.exp(-1 * weighted_sum))


def forward_propagate(network, inputs):
    """Propaga gli input strato per strato fino a ottenere la previsione finale."""
    layer_inputs = list(inputs)

    for layer in network:
        layer_data = network[layer]
        layer_outputs = []

        for layer_node in layer_data:
            node_data = layer_data[layer_node]
            node_output = node_activation(
                compute_weighted_sum(layer_inputs, node_data['weights'], node_data['bias'])
            )
            layer_outputs.append(np.around(node_output[0], decimals=4))

        if layer != 'output':
            print('Output dello strato nascosto {}: {}'.format(layer.split('_')[1], layer_outputs))

        layer_inputs = layer_outputs

    return layer_outputs

print('Funzioni definite: initialize_network, compute_weighted_sum, node_activation, forward_propagate')

## Proviamo la rete con due architetture diverse

Per prima cosa, `small_network`: 5 input, 3 strati nascosti (3, 2 e 3 nodi) e 1 output.

In [ ]:
np.random.seed(0)  # per rendere la demo riproducibile in classe

small_network = initialize_network(
    num_inputs=5,
    num_hidden_layers=3,
    num_nodes_hidden=[3, 2, 3],
    num_nodes_output=1
)

inputs = np.around(np.random.uniform(size=5), decimals=2)
print('Input:', inputs)

predictions = forward_propagate(small_network, inputs)
print('Previsione finale di small_network:', predictions)

In [ ]:
# Seconda rete: my_network, stesso numero di input ma un'altra architettura
# (2, 3 e 2 nodi nascosti, e 3 nodi di output invece di 1)
my_network = initialize_network(
    num_inputs=5,
    num_hidden_layers=3,
    num_nodes_hidden=[2, 3, 2],
    num_nodes_output=3
)

inputs = np.around(np.random.uniform(size=5), decimals=2)
print('Input:', inputs)

predictions = forward_propagate(my_network, inputs)
print('Previsione finale di my_network:', predictions)

## Ripasso teorico veloce: funzioni di attivazione e vanishing gradient

- **Sigmoide** `σ(z) = 1/(1+e^-z)`: output tra 0 e 1, derivata `σ'(z) = σ(z)(1-σ(z))`. Problema: per `z` molto grande o molto negativo, la derivata si avvicina a 0 → **vanishing gradient** (il gradiente "sparisce" propagandosi all'indietro in reti profonde).
- **ReLU** `f(x) = max(0, x)`: derivata 1 se `x>0`, 0 se `x≤0`. Mitiga il vanishing gradient perché la sua derivata non si appiattisce nella zona positiva. Rischio: "neuroni morti" se l'input è sempre negativo.

Grafichiamo entrambe le funzioni insieme alle loro derivate per vedere il problema con i nostri occhi.

In [ ]:
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return np.where(z > 0, 1, 0)

z = np.linspace(-10, 10, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(z, sigmoid(z), label='Sigmoide')
axes[0].plot(z, sigmoid_derivative(z), label='Derivata')
axes[0].set_title('Sigmoide: la derivata si appiattisce agli estremi')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(z, relu(z), label='ReLU')
axes[1].plot(z, relu_derivative(z), label='Derivata')
axes[1].set_title('ReLU: la derivata resta a 1 per z > 0')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Riepilogo della lezione

- Forward propagation = somma pesata + attivazione, concatenate strato per strato.
- Con NumPy puro puoi rappresentare una rete generica come un dizionario di pesi e bias per ogni nodo.
- La sigmoide soffre di vanishing gradient ai suoi estremi; ReLU lo mitiga perché la sua derivata è costante nella zona positiva.
- Con pesi casuali la rete già prevede qualcosa, ma non ha ancora imparato nulla — questo è il backpropagation, argomento della lezione 2.

**Per approfondire:**
- Capitolo del manuale: [`manual/02_forward_propagation.md`](../manual/02_forward_propagation_IT.md) e [`manual/03_funciones_activacion.md`](../manual/03_funciones_activacion_IT.md)
- Notebook di pratica di questa lezione: [`notebooks_practica/clase01_practica_forward_propagation_activaciones.ipynb`](../notebooks_practica/clase01_practica_forward_propagation_activaciones_IT.ipynb)